# VAD Evaluation — Sarvam AI Interview Challenge

Batch-run the Denoiser + WebRTC (GMM) pipeline across `sample_data/sample_audio/`,
measure speech ratios and wall-clock latency, and plot a sample timeline.

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "examples" / "sarvam-vad-challenge").exists():
    # Notebook may be launched from notebooks/
    REPO_ROOT = Path.cwd().parent

CHALLENGE_DIR = REPO_ROOT / "examples" / "sarvam-vad-challenge"
AUDIO_DIR = REPO_ROOT / "sample_data" / "sample_audio"
sys.path.insert(0, str(CHALLENGE_DIR))

from pipeline import VADPipeline

print("Repo:", REPO_ROOT)
print("Audio dir exists:", AUDIO_DIR.exists())
print("WAV count:", len(list(AUDIO_DIR.glob("*.wav"))) if AUDIO_DIR.exists() else 0)

In [ ]:
pipeline = VADPipeline(aggressiveness=3)
rows = []

wavs = sorted(AUDIO_DIR.glob("*.wav"))
assert wavs, f"No WAVs in {AUDIO_DIR}. Run: python examples/sarvam-vad-challenge/fetch_samples.py"

for path in wavs:
    t0 = time.perf_counter()
    result = pipeline.process_file(str(path))
    elapsed_ms = (time.perf_counter() - t0) * 1000
    rows.append(
        {
            "file": path.name,
            "duration_sec": len(result.clean_audio) / result.sample_rate,
            "speech_ratio": result.speech_ratio,
            "speech_pct": result.speech_ratio * 100,
            "n_segments": len(result.speech_segments),
            "latency_ms": elapsed_ms,
        }
    )

df = pd.DataFrame(rows)
df.describe()

In [ ]:
display(df.head(10))
print("Mean speech %:", round(df.speech_pct.mean(), 2))
print("Mean latency ms:", round(df.latency_ms.mean(), 2))
print("p95 latency ms:", round(df.latency_ms.quantile(0.95), 2))

In [ ]:
sample = wavs[0]
result = pipeline.process_file(str(sample))
frame_ms = 30
times = np.arange(len(result.speech_flags)) * (frame_ms / 1000.0)
audio_t = np.arange(len(result.clean_audio)) / result.sample_rate

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(audio_t, result.clean_audio, linewidth=0.6, color="#1f4e5f")
axes[0].set_ylabel("Amplitude")
axes[0].set_title(f"Denoised waveform — {sample.name}")

axes[1].fill_between(times, result.speech_flags, step="pre", alpha=0.7, color="#c45c26")
axes[1].set_ylabel("Speech")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title("WebRTC VAD speech flags (30 ms frames)")
plt.tight_layout()
plt.show()

## A/B: denoise on vs off

Useful talking point for the interview — quantify how much spectral gating
changes detected speech ratio on the same set.

In [ ]:
compare = []
for path in wavs[:20]:
    with_denoise = pipeline.process_file(str(path), denoise=True)
    raw = pipeline.process_file(str(path), denoise=False)
    compare.append(
        {
            "file": path.name,
            "speech_pct_denoised": with_denoise.speech_ratio * 100,
            "speech_pct_raw": raw.speech_ratio * 100,
            "delta_pp": (with_denoise.speech_ratio - raw.speech_ratio) * 100,
        }
    )

cmp_df = pd.DataFrame(compare)
display(cmp_df.describe())
cmp_df.head()